<a href="https://colab.research.google.com/github/AlperYildirim1/Oppenheim-Lim-Neural-Networks/blob/main/Oppenheim_%26_Lim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# OPPENHEIM & LIM ON PRISM2D — STANDALONE NOTEBOOK
# ============================================================================
#   /content/drive/MyDrive/prism2d_runs/prism2d/best.pt   (and last.pt)
#   /content/drive/MyDrive/prism2d_runs/imagenet100_hf    (HF dataset cache)
# Outputs (CSV + figures) are written back into the run dir on Drive.
# Eval-only. ~30-45 min on A100 for the full sweep at 2500 pairs.

# ============================================================================
# CELL 1 — SETUP (mount drive, install, paths)
# ============================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# !pip -q install datasets

import os, csv, math
import torch
import torch.nn as nn
import torch.nn.functional as F

OL_CFG = dict(
    drive_root  = "/content/drive/MyDrive/prism2d_runs",
    run_name    = "prism2d",
    ckpt        = "best.pt",          # also rerun with "last.pt"
    data_local  = "/content/data",
    # --- must MATCH the trained model (CELL T4 used depth=10) ---
    img_size=224, patch_size=16, in_chans=3, num_classes=100,
    d_model=256, depth=10, dropout=0.1,
    fft_mode="spatial", activation="modrelu", filter_hidden=64,
    # --- experiment ---
    n_pairs=2500, batch_pairs=64, num_workers=8, seed=42,
    amp_dtype=torch.bfloat16,
)
OL_CFG["run_dir"] = os.path.join(OL_CFG["drive_root"], OL_CFG["run_name"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")


# ============================================================================
# CELL 2 — MODEL (verbatim PRISM2D definition + built-in swap_mode support)
# ============================================================================
# Identical module/attribute names as training code -> state_dict loads 1:1.
# Only addition: the intervention hook inside GatedHarmonicConvolution2D.

class RobustPhaseNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        mag = torch.abs(x)
        rms = torch.sqrt(torch.mean(mag ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

class ModReLU(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.zeros(features))
    def forward(self, z):
        mag = torch.abs(z)
        return F.relu(mag + self.b) * (z / (mag + 1e-6))

class CReLU(nn.Module):
    def __init__(self, features=None):
        super().__init__()
    def forward(self, z):
        return torch.complex(F.relu(z.real), F.relu(z.imag))

class ComplexDropout(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p
    def forward(self, z):
        if not self.training or self.p == 0.0:
            return z
        mask = F.dropout(torch.ones_like(z.real), self.p, self.training)
        return z * mask

class ComplexToRealBridge(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x_complex):
        cat = torch.cat([x_complex.real, x_complex.imag], dim=-1)
        return self.norm(self.proj(cat))

class DynamicRoSE2D(nn.Module):
    def __init__(self, d_model, patch_size=16, in_chans=3, max_period=10000.0):
        super().__init__()
        self.d_model = d_model
        self.patch_embed = nn.Conv2d(in_chans, d_model,
                                     kernel_size=patch_size, stride=patch_size)
        self.adapter = nn.Linear(d_model, d_model * 2)
        self.rotation_predictor = nn.Linear(d_model, d_model * 2)
        half = d_model // 2
        freqs = torch.exp(torch.arange(0, half, dtype=torch.float32)
                          * -(math.log(max_period) / half))
        self.register_buffer("freqs", freqs)

    def _static_rotation(self, H, W, device):
        rows = torch.arange(H, device=device).float()
        cols = torch.arange(W, device=device).float()
        ang_r = torch.outer(rows, self.freqs)[:, None, :].expand(H, W, -1)
        ang_c = torch.outer(cols, self.freqs)[None, :, :].expand(H, W, -1)
        angles = torch.cat([ang_r, ang_c], dim=-1)
        return torch.polar(torch.ones_like(angles), angles)

    def forward(self, images):
        x = self.patch_embed(images)
        real_base = x.permute(0, 2, 3, 1).contiguous()
        B, H, W, D = real_base.shape
        cp = self.adapter(real_base).float()
        z_t = torch.complex(cp[..., :D], cp[..., D:])
        rr = self.rotation_predictor(real_base).float()
        rx, ry = rr.chunk(2, dim=-1)
        rmag = torch.sqrt(rx ** 2 + ry ** 2 + 1e-6)
        dyn_rot = torch.complex(rx / rmag, ry / rmag)
        static_rot = self._static_rotation(H, W, images.device)
        return z_t * static_rot.unsqueeze(0) * dyn_rot, real_base

class NeuralFilter2D(nn.Module):
    def __init__(self, d_model, hidden_dim=64):
        super().__init__()
        self.d_model = d_model
        nf = hidden_dim // 2
        freqs = torch.exp(torch.arange(0, nf, dtype=torch.float32)
                          * -(math.log(10000.0) / nf))
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, d_model * 2))
        self._cache_key, self._cache_val = None, None

    def train(self, mode: bool = True):
        self._cache_key, self._cache_val = None, None   # the cache fix
        return super().train(mode)

    def _embed(self, t):
        ang = t * self.freqs
        return torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1)

    def forward(self, H, W, device):
        key = (H, W, device, self.training)
        if (not self.training) and self._cache_key == key:
            return self._cache_val
        r = torch.linspace(0, 1, steps=H, device=device).unsqueeze(-1)
        c = torch.linspace(0, 1, steps=W, device=device).unsqueeze(-1)
        er, ec = self._embed(r), self._embed(c)
        grid = torch.cat([er[:, None, :].expand(H, W, -1),
                          ec[None, :, :].expand(H, W, -1)], dim=-1)
        out = self.mlp(grid).float().view(H, W, self.d_model, 2)
        h = torch.complex(out[..., 0], out[..., 1])
        if not self.training:
            self._cache_key, self._cache_val = key, h
        return h

# ---- swap primitives (used by the layer below) ----
def pair_swap_phase(z):
    """Pairs (2i, 2i+1): out[2i]=|A|*e^{i angle(B)}, out[2i+1]=|B|*e^{i angle(A)}"""
    mag = z.abs()
    ph = z / (mag + 1e-9)
    ph_sw = torch.empty_like(ph)
    ph_sw[0::2], ph_sw[1::2] = ph[1::2], ph[0::2]
    return mag * ph_sw

def random_phase(z):
    theta = (torch.rand_like(z.real) * 2 - 1) * math.pi
    return z.abs() * torch.polar(torch.ones_like(z.real), theta)

def unit_mag(z):
    return z / (z.abs() + 1e-9)

def apply_swap(z, mode):
    return {"swap": pair_swap_phase,
            "rand_phase": random_phase,
            "unit_mag": unit_mag}[mode](z)

class GatedHarmonicConvolution2D(nn.Module):
    def __init__(self, d_model, dropout=0.1, fft_mode="spatial",
                 activation="modrelu", filter_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.fft_mode = fft_mode
        self.neural_filter = NeuralFilter2D(d_model, hidden_dim=filter_hidden)
        self.value_proj = nn.Linear(d_model * 2, d_model * 2)
        self.gate_proj  = nn.Linear(d_model * 2, d_model * 2)
        self.out_real   = nn.Linear(d_model, d_model)
        self.out_imag   = nn.Linear(d_model, d_model)
        self.activation = ModReLU(d_model) if activation == "modrelu" else CReLU(d_model)
        self.norm = RobustPhaseNorm(d_model)
        self.dropout = ComplexDropout(dropout)
        self.swap_mode = None          # O&L intervention point (eval-only use)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x)
        B, H, W, D = x_norm.shape
        x_freq = torch.fft.fft2(x_norm, dim=(1, 2), norm="ortho")
        if self.swap_mode is not None:
            x_freq = apply_swap(x_freq, self.swap_mode)
        h = self.neural_filter(H, W, x.device).unsqueeze(0)
        x_time = torch.fft.ifft2(x_freq * h, dim=(1, 2), norm="ortho")
        x_cat = torch.cat([x_time.real, x_time.imag], dim=-1)
        v_raw = self.value_proj(x_cat).float()
        g_raw = self.gate_proj(x_cat).float()
        Z_value = torch.complex(v_raw[..., :D], v_raw[..., D:])
        Z_gate  = torch.complex(g_raw[..., :D], g_raw[..., D:])
        x_act = self.activation(Z_value * Z_gate)
        out = torch.complex(
            (self.out_real(x_act.real) - self.out_imag(x_act.imag)).float(),
            (self.out_real(x_act.imag) + self.out_imag(x_act.real)).float())
        return self.dropout(out) + residual

class PRISM2D(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=100,
                 d_model=256, depth=10, dropout=0.1, fft_mode="spatial",
                 activation="modrelu", filter_hidden=64, grad_ckpt=False):
        super().__init__()
        self.grad_ckpt = grad_ckpt
        self.rose = DynamicRoSE2D(d_model, patch_size, in_chans)
        self.layers = nn.ModuleList([
            GatedHarmonicConvolution2D(d_model, dropout, fft_mode,
                                       activation, filter_hidden)
            for _ in range(depth)])
        self.final_norm = RobustPhaseNorm(d_model)
        self.bridge = ComplexToRealBridge(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward_features(self, images):
        x, _ = self.rose(images)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        x = self.bridge(x)
        return x.mean(dim=(1, 2))

    def forward(self, images):
        return self.head(self.forward_features(images))


# ============================================================================
# CELL 3 — LOAD CHECKPOINT
# ============================================================================

def load_model(cfg=OL_CFG):
    model = PRISM2D(
        img_size=cfg["img_size"], patch_size=cfg["patch_size"],
        in_chans=cfg["in_chans"], num_classes=cfg["num_classes"],
        d_model=cfg["d_model"], depth=cfg["depth"], dropout=cfg["dropout"],
        fft_mode=cfg["fft_mode"], activation=cfg["activation"],
        filter_hidden=cfg["filter_hidden"]).to(DEVICE)
    path = os.path.join(cfg["run_dir"], cfg["ckpt"])
    ck = torch.load(path, map_location="cpu")
    missing, unexpected = model.load_state_dict(ck["model"], strict=True), None
    model.eval()
    print(f"loaded {path} (saved after epoch {ck.get('epoch','?')}, "
          f"best {ck.get('best_acc', float('nan')):.2f}%)")
    return model

model = load_model()


# ============================================================================
# CELL 4 — DATA (Drive cache -> local SSD, paired different-class val set)
# ============================================================================
from datasets import load_from_disk
from torchvision import transforms as T

def get_dataset(cfg=OL_CFG):
    drive_cache = os.path.join(cfg["drive_root"], "imagenet100_hf")
    local_cache = os.path.join(cfg["data_local"], "imagenet100_hf")
    if not os.path.exists(local_cache):
        print("Copying dataset Drive -> local SSD...")
        os.makedirs(cfg["data_local"], exist_ok=True)
        os.system(f"cp -r '{drive_cache}' '{local_cache}'")
    ds = load_from_disk(local_cache)
    print(f"val: {len(ds['validation'])}")
    return ds

VAL_TF = T.Compose([
    T.Resize(int(OL_CFG["img_size"] / 0.9), interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(OL_CFG["img_size"]),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

class PairedVal(torch.utils.data.Dataset):
    """Different-class val pairs; batch layout [A0,B0,A1,B1,...]."""
    def __init__(self, hf_val, n_pairs, seed):
        g = torch.Generator().manual_seed(seed)
        labels = hf_val["label"]
        idx = torch.randperm(len(hf_val), generator=g).tolist()
        self.pairs, i = [], 0
        while len(self.pairs) < n_pairs and i + 1 < len(idx):
            a, b = idx[i], idx[i + 1]
            if labels[a] != labels[b]:
                self.pairs.append((a, b)); i += 2
            else:
                i += 1
        self.ds = hf_val
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, k):
        a, b = self.pairs[k]
        ea, eb = self.ds[a], self.ds[b]
        return (VAL_TF(ea["image"].convert("RGB")), ea["label"],
                VAL_TF(eb["image"].convert("RGB")), eb["label"])

def collate_pairs(batch):
    xs, ys = [], []
    for xa, ya, xb, yb in batch:
        xs += [xa, xb]; ys += [ya, yb]
    return torch.stack(xs), torch.tensor(ys)

ds = get_dataset()
pair_loader = torch.utils.data.DataLoader(
    PairedVal(ds["validation"], OL_CFG["n_pairs"], OL_CFG["seed"]),
    batch_size=OL_CFG["batch_pairs"], shuffle=False,
    num_workers=OL_CFG["num_workers"], pin_memory=True,
    collate_fn=collate_pairs)

# class names for the qualitative figure (fallback: indices)
try:
    CLASS_NAMES = ds["validation"].features["label"].names
except Exception:
    CLASS_NAMES = [str(i) for i in range(OL_CFG["num_classes"])]


# ============================================================================
# CELL 5 — RUNNERS & SCORING
# ============================================================================

def set_branch_modes(model, layer_ids, mode):
    for i, layer in enumerate(model.layers):
        layer.swap_mode = mode if i in layer_ids else None

def clear_modes(model):
    set_branch_modes(model, set(), None)

@torch.no_grad()
def forward_stream_swap(model, images, swap_at, mode="swap"):
    x, _ = model.rose(images)
    for i, layer in enumerate(model.layers):
        if i == swap_at:
            x = apply_swap(x, mode)
        x = layer(x)
    return model.head(model.bridge(model.final_norm(x)).mean(dim=(1, 2)))

@torch.no_grad()
def forward_pixel_swap(model, images):
    f = torch.fft.fft2(images.float(), dim=(-2, -1), norm="ortho")
    chim = torch.fft.ifft2(pair_swap_phase(f), dim=(-2, -1), norm="ortho").real
    return model(chim)

@torch.no_grad()
def score_chimeras(logits, y):
    pred = logits.argmax(-1)
    phase_donor = y.clone()
    phase_donor[0::2], phase_donor[1::2] = y[1::2], y[0::2]
    return dict(follow_phase=(pred == phase_donor).sum().item(),
                follow_mag=(pred == y).sum().item(),
                other=((pred != phase_donor) & (pred != y)).sum().item(),
                n=y.numel())

@torch.no_grad()
def score_self(logits, y):
    pred = logits.argmax(-1)
    return dict(follow_phase=0, follow_mag=(pred == y).sum().item(),
                other=(pred != y).sum().item(), n=y.numel())

@torch.no_grad()
def run_condition(fwd, scorer):
    agg = dict(follow_phase=0, follow_mag=0, other=0, n=0)
    for x, y in pair_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast("cuda", dtype=OL_CFG["amp_dtype"],
                            enabled=DEVICE.type == "cuda"):
            logits = fwd(x)
        for k, v in scorer(logits.float(), y).items():
            agg[k] += v
    n = agg["n"]
    return dict(follow_phase=round(100 * agg["follow_phase"] / n, 2),
                follow_mag=round(100 * agg["follow_mag"] / n, 2),
                other=round(100 * agg["other"] / n, 2), n=n)


# ============================================================================
# CELL 6 — MAIN EXPERIMENT (full sweep -> CSV on Drive)
# ============================================================================

def ol_experiment(model, cfg=OL_CFG):
    depth = len(model.layers)
    rows = []
    def log(name, res):
        rows.append(dict(condition=name, **res)); print(rows[-1])

    clear_modes(model)
    log("baseline", run_condition(lambda x: model(x), score_self))
    log("pixel_swap", run_condition(lambda x: forward_pixel_swap(model, x),
                                    score_chimeras))

    for l in range(depth):                       # single-layer branch swap
        set_branch_modes(model, {l}, "swap")
        log(f"branch_single_L{l}",
            run_condition(lambda x: model(x), score_chimeras))

    for l in range(depth):                       # cumulative branch swap
        set_branch_modes(model, set(range(l + 1)), "swap")
        log(f"branch_cum_L0-{l}",
            run_condition(lambda x: model(x), score_chimeras))

    clear_modes(model)
    for l in range(depth):                       # stream swap
        log(f"stream_L{l}",
            run_condition(lambda x, _l=l: forward_stream_swap(model, x, _l),
                          score_chimeras))

    for mode in ("rand_phase", "unit_mag"):      # controls (self-label)
        set_branch_modes(model, set(range(depth)), mode)
        log(f"control_{mode}_all", run_condition(lambda x: model(x), score_self))
    clear_modes(model)

    out = os.path.join(cfg["run_dir"], f"ol_results_{cfg['ckpt'].split('.')[0]}.csv")
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print("saved ->", out)
    return rows

rows = ol_experiment(model)


# ============================================================================
# CELL 7 — "THE MODEL AS OBSERVER" QUALITATIVE FIGURE
# ============================================================================
# The 1981 demo, but the caption is written by the classifier: four panels,
# model's top-1 under each. This is how you say "it looks like Fourier"
# in a classification model — the model says it.

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denorm(x):
    return (x.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

@torch.no_grad()
def observer_figure(model, pair_idx=0, save="ol_observer.png", cfg=OL_CFG):
    import matplotlib.pyplot as plt
    pv = pair_loader.dataset
    xa, ya, xb, yb = pv[pair_idx]
    x = torch.stack([xa, xb]).to(DEVICE)
    f = torch.fft.fft2(x.float(), dim=(-2, -1), norm="ortho")
    chim = torch.fft.ifft2(pair_swap_phase(f), dim=(-2, -1), norm="ortho").real
    batch = torch.cat([x, chim])                 # [A, B, |A|∠B, |B|∠A]
    pred = model(batch).argmax(-1).tolist()
    titles = [f"A: {CLASS_NAMES[ya]}\nmodel: {CLASS_NAMES[pred[0]]}",
              f"B: {CLASS_NAMES[yb]}\nmodel: {CLASS_NAMES[pred[1]]}",
              f"|A| + ∠B\nmodel: {CLASS_NAMES[pred[2]]}",
              f"|B| + ∠A\nmodel: {CLASS_NAMES[pred[3]]}"]
    fig, ax = plt.subplots(1, 4, figsize=(14, 4))
    for a, img, t in zip(ax, batch, titles):
        a.imshow(denorm(img)); a.set_title(t, fontsize=9); a.axis("off")
    plt.tight_layout()
    out = os.path.join(cfg["run_dir"], save)
    plt.savefig(out, dpi=200); plt.show()
    print("saved ->", out)

observer_figure(model, pair_idx=0)


# ============================================================================
# CELL 8 — DOMINANCE CURVES (plot from rows)
# ============================================================================

def plot_curves(rows, cfg=OL_CFG):
    import matplotlib.pyplot as plt
    def series(prefix):
        sel = [r for r in rows if r["condition"].startswith(prefix)]
        return ([r["follow_phase"] for r in sel],
                [r["follow_mag"] for r in sel])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, prefix, title in zip(
            axes, ["branch_single_L", "branch_cum_L", "stream_L"],
            ["Single-layer branch swap", "Cumulative branch swap 0..l",
             "Stream swap at layer l"]):
        fp, fm = series(prefix)
        ax.plot(fp, "o-", label="follow phase donor")
        ax.plot(fm, "s-", label="follow magnitude donor")
        ax.axhline(1.0, ls=":", c="gray", label="chance (1%)")
        ax.set_title(title); ax.set_xlabel("layer l"); ax.set_ylim(0, 100)
    axes[0].set_ylabel("% of chimeras"); axes[0].legend()
    plt.tight_layout()
    out = os.path.join(cfg["run_dir"], "ol_dominance_curves.png")
    plt.savefig(out, dpi=200); plt.show()
    print("saved ->", out)

plot_curves(rows)

In [ ]:
# ============================================================================
# O&L ON GFNET
# ============================================================================
# session (OL_CFG, DEVICE, pair_loader, scorers, apply_swap, pair_swap_phase).
# Loads:  /content/drive/MyDrive/prism2d_runs/gfnet_ti/best.pt
# Writes: ol_results_gfnet_<ckpt>.csv into the gfnet_ti run dir.

# ============================================================================
# CELL G1 — GFNET (vendored to match the trained state_dict; swap built in)
# ============================================================================
import os, csv, math
import torch
import torch.nn as nn

GF_CFG = dict(OL_CFG)
GF_CFG["run_name"] = "gfnet_ti"
GF_CFG["run_dir"] = os.path.join(GF_CFG["drive_root"], GF_CFG["run_name"])
GF_CFG["depth"] = 12                       # gfnet_ti: embed_dim=256, depth=12

class DropPathStub(nn.Module):             # eval-only; no params, no-op
    def __init__(self, *a, **k): super().__init__()
    def forward(self, x): return x

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

class GlobalFilter(nn.Module):
    def __init__(self, dim, h=14, w=8):
        super().__init__()
        self.complex_weight = nn.Parameter(
            torch.randn(h, w, dim, 2, dtype=torch.float32) * 0.02)
        self.w = w; self.h = h
        self.swap_mode = None              # O&L intervention point

    def forward(self, x, spatial_size=None):
        B, N, C = x.shape
        a = b = int(math.sqrt(N)) if spatial_size is None else spatial_size
        x = x.view(B, a, b, C).to(torch.float32)
        x = torch.fft.rfft2(x, dim=(1, 2), norm='ortho')
        if self.swap_mode is not None:     # swap before weight == after (h
            x = apply_swap(x, self.swap_mode)   # multiplies both factors)
        weight = torch.view_as_complex(self.complex_weight)
        x = x * weight
        x = torch.fft.irfft2(x, s=(a, b), dim=(1, 2), norm='ortho')
        return x.reshape(B, N, C)

class GFBlock(nn.Module):
    def __init__(self, dim, mlp_ratio=4., drop=0., drop_path=0.,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm, h=14, w=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.filter = GlobalFilter(dim, h=h, w=w)
        self.drop_path = DropPathStub()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(dim, int(dim * mlp_ratio), act_layer=act_layer, drop=drop)
    def forward(self, x):
        return x + self.drop_path(self.mlp(self.norm2(self.filter(self.norm1(x)))))

class GFPatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class GFNet(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=100,
                 embed_dim=256, depth=12, mlp_ratio=4., drop_rate=0.):
        super().__init__()
        norm_layer = lambda d: nn.LayerNorm(d, eps=1e-6)
        self.patch_embed = GFPatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)
        h = img_size // patch_size
        w = h // 2 + 1
        self.blocks = nn.ModuleList([
            GFBlock(embed_dim, mlp_ratio, drop_rate, 0., norm_layer=norm_layer,
                    h=h, w=w) for _ in range(depth)])
        self.norm = norm_layer(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x).mean(1))

def load_gfnet(cfg=GF_CFG):
    model = GFNet(img_size=cfg["img_size"], num_classes=cfg["num_classes"],
                  embed_dim=256, depth=cfg["depth"]).to(DEVICE)
    path = os.path.join(cfg["run_dir"], cfg["ckpt"])
    ck = torch.load(path, map_location="cpu")
    model.load_state_dict(ck["model"], strict=True)
    model.eval()
    print(f"loaded {path} (epoch {ck.get('epoch','?')}, "
          f"best {ck.get('best_acc', float('nan')):.2f}%)")
    return model

gf = load_gfnet()


# ============================================================================
# CELL G2 — GFNET RUNNERS
# ============================================================================

def gf_set_branch(model, layer_ids, mode):
    for i, blk in enumerate(model.blocks):
        blk.filter.swap_mode = mode if i in layer_ids else None

def gf_clear(model):
    gf_set_branch(model, set(), None)

def sign_swap(x):
    """Degenerate real analog of channel-phase transplant:
    swap sign(x) between pairs, keep |x|. Real 'phase' is 0/pi."""
    s, m = torch.sign(x), x.abs()
    s_sw = torch.empty_like(s)
    s_sw[0::2], s_sw[1::2] = s[1::2], s[0::2]
    return m * s_sw

@torch.no_grad()
def gf_forward_stream_spectral(model, images, swap_at, mode="swap"):
    """FFT the full real stream over the token grid at block `swap_at`,
    swap (or control), inverse, continue. The GFNet analog of PRISM's
    stream spectral swap."""
    x = model.patch_embed(images)
    x = model.pos_drop(x + model.pos_embed)
    B, N, C = x.shape
    a = b = int(math.sqrt(N))
    for i, blk in enumerate(model.blocks):
        if i == swap_at:
            g = x.view(B, a, b, C).float()
            gf_ = torch.fft.rfft2(g, dim=(1, 2), norm="ortho")
            g = torch.fft.irfft2(apply_swap(gf_, mode), s=(a, b),
                                 dim=(1, 2), norm="ortho")
            x = g.reshape(B, N, C).to(x.dtype)
        x = blk(x)
    return model.head(model.norm(x).mean(1))

@torch.no_grad()
def gf_forward_sign_swap(model, images, swap_at):
    """Token-space sign transplant at block `swap_at` (the only 'feature
    phase' a real stream has)."""
    x = model.patch_embed(images)
    x = model.pos_drop(x + model.pos_embed)
    for i, blk in enumerate(model.blocks):
        if i == swap_at:
            x = sign_swap(x)
        x = blk(x)
    return model.head(model.norm(x).mean(1))

@torch.no_grad()
def gf_forward_pixel_swap(model, images):
    f = torch.fft.fft2(images.float(), dim=(-2, -1), norm="ortho")
    chim = torch.fft.ifft2(pair_swap_phase(f), dim=(-2, -1), norm="ortho").real
    return model(chim)


# ============================================================================
# CELL G3 — GFNET EXPERIMENT (mirrors the PRISM sweep; reuses run_condition)
# ============================================================================

def ol_experiment_gfnet(model, cfg=GF_CFG):
    depth = len(model.blocks)
    rows = []
    def log(name, res):
        rows.append(dict(condition=name, **res)); print(rows[-1])

    gf_clear(model)
    log("baseline", run_condition(lambda x: model(x), score_self))
    log("pixel_swap", run_condition(lambda x: gf_forward_pixel_swap(model, x),
                                    score_chimeras))

    for l in range(depth):                       # branch swap (rfft2 domain)
        gf_set_branch(model, {l}, "swap")
        log(f"branch_single_L{l}",
            run_condition(lambda x: model(x), score_chimeras))

    for l in range(depth):                       # cumulative branch swap
        gf_set_branch(model, set(range(l + 1)), "swap")
        log(f"branch_cum_L0-{l}",
            run_condition(lambda x: model(x), score_chimeras))

    gf_clear(model)
    for l in range(depth):                       # stream SPECTRAL swap
        log(f"stream_spec_L{l}",
            run_condition(lambda x, _l=l: gf_forward_stream_spectral(model, x, _l),
                          score_chimeras))

    for l in range(depth):                       # sign swap (degenerate analog)
        log(f"sign_swap_L{l}",
            run_condition(lambda x, _l=l: gf_forward_sign_swap(model, x, _l),
                          score_chimeras))

    # controls at mid-depth, single-shot (uncompounded)
    mid = depth // 2
    for mode in ("rand_phase", "unit_mag"):
        log(f"control_stream_{mode}_L{mid}",
            run_condition(lambda x, _m=mode: gf_forward_stream_spectral(
                model, x, mid, _m), score_self))

    out = os.path.join(cfg["run_dir"],
                       f"ol_results_gfnet_{cfg['ckpt'].split('.')[0]}.csv")
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print("saved ->", out)
    return rows

gf_rows = ol_experiment_gfnet(gf)


# ============================================================================
# CELL G4 — PRISM vs GFNET COMPARISON FIGURE
# ============================================================================
# Run after both `rows` (PRISM) and `gf_rows` exist in the session.

def comparison_figure(prism_rows, gfnet_rows, cfg=OL_CFG):
    import matplotlib.pyplot as plt
    def fp(rows, prefix):
        return [r["follow_phase"] for r in rows
                if r["condition"].startswith(prefix)]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(fp(prism_rows, "stream_L"), "o-",
            label="PRISM2D: channel-phase transplant")
    ax.plot(fp(gfnet_rows, "stream_spec_L"), "s-",
            label="GFNet: spatial-spectral phase transplant")
    ax.plot(fp(gfnet_rows, "sign_swap_L"), "^--",
            label="GFNet: sign transplant (real 'phase')")
    ax.axhline(1.0, ls=":", c="gray", label="chance (1%)")
    ax.set_xlabel("intervention layer l")
    ax.set_ylabel("% follow phase donor")
    ax.set_ylim(0, 100); ax.legend(fontsize=8)
    ax.set_title("Identity follows phase: complex vs real spectral models")
    plt.tight_layout()
    out = os.path.join(cfg["run_dir"], "ol_prism_vs_gfnet.png")
    plt.savefig(out, dpi=200); plt.show()
    print("saved ->", out)

# comparison_figure(rows, gf_rows)

In [ ]:
# ============================================================================
# BORING-vs-INTERESTING DIAGNOSTICS — APPEND TO THE SAME SESSION
# ============================================================================
# Requires from earlier cells: OL_CFG, DEVICE, pair_loader, run_condition,
# score_self, `model` (PRISM2D), `gf` (GFNet).
#
# Boring hypothesis H0: late-layer magnitudes are image-independent, so the
#   sign/phase chimera is geometrically ~= the phase donor; following it is
#   trivial.
# Interesting H1: magnitudes remain image-specific, the chimera is far from
#   the donor, yet the readout still follows sign/phase.
#
# Test 1 (geometry): per layer, over ~256 pairs:
#   mag_corr(A,B), cos(chimera,B), cos(chimera,A), cos(A,B)
# Test 2 (causal): mean-magnitude substitution x -> mean|x| * sign(x)
#   (deletes ALL image-specific magnitude info, imports none). Self-label
#   accuracy per layer.

import itertools, os, csv
import torch
import torch.nn.functional as F

N_GEOM_BATCHES = 4          # 4 x 64 pairs = 256 pairs for geometry stats
EPS = 1e-8

# ============================================================================
# CELL D1 — METRICS & STATE CAPTURE
# ============================================================================

def pearson(a, b):                          # a,b: [P, F] per-pair flattened
    a = a - a.mean(-1, keepdim=True)
    b = b - b.mean(-1, keepdim=True)
    return ((a * b).sum(-1) /
            (a.norm(dim=-1) * b.norm(dim=-1) + EPS))

def cos_real(a, b):
    return (a * b).sum(-1) / (a.norm(dim=-1) * b.norm(dim=-1) + EPS)

def cos_complex(a, b):
    num = (a * b.conj()).sum(-1).real
    return num / (a.abs().pow(2).sum(-1).sqrt()
                  * b.abs().pow(2).sum(-1).sqrt() + EPS)

@torch.no_grad()
def gf_capture(model, images):
    """List of stream states (input to each block). [B, N, C] real."""
    states = []
    x = model.patch_embed(images)
    x = model.pos_drop(x + model.pos_embed)
    for blk in model.blocks:
        states.append(x.clone())
        x = blk(x)
    return states

@torch.no_grad()
def prism_capture(model, images):
    """List of stream states (input to each layer). [B, H, W, D] complex."""
    states = []
    x, _ = model.rose(images)
    for layer in model.layers:
        states.append(x.clone())
        x = layer(x)
    return states

@torch.no_grad()
def geometry_table(model, capture_fn, is_complex):
    depth = len(model.layers) if hasattr(model, "layers") else len(model.blocks)
    acc = [dict(mag_corr=[], cos_chB=[], cos_chA=[], cos_AB=[])
           for _ in range(depth)]
    for x, y in itertools.islice(pair_loader, N_GEOM_BATCHES):
        x = x.to(DEVICE)
        with torch.autocast("cuda", dtype=OL_CFG["amp_dtype"],
                            enabled=DEVICE.type == "cuda"):
            states = capture_fn(model, x)
        for l, s in enumerate(states):
            s = s.float() if not is_complex else s
            A = s[0::2].flatten(1)          # [P, F]
            B = s[1::2].flatten(1)
            if is_complex:
                chim = A.abs() * (B / (B.abs() + EPS))      # |A| * e^{i angle B}
                acc[l]["cos_chB"].append(cos_complex(chim, B))
                acc[l]["cos_chA"].append(cos_complex(chim, A))
                acc[l]["cos_AB"].append(cos_complex(A, B))
            else:
                chim = A.abs() * torch.sign(B)              # |A| * sign(B)
                acc[l]["cos_chB"].append(cos_real(chim, B))
                acc[l]["cos_chA"].append(cos_real(chim, A))
                acc[l]["cos_AB"].append(cos_real(A, B))
            acc[l]["mag_corr"].append(pearson(A.abs(), B.abs()))
    rows = []
    for l, d in enumerate(acc):
        rows.append(dict(layer=l, **{k: round(torch.cat(v).mean().item(), 3)
                                     for k, v in d.items()}))
        print(rows[-1])
    return rows

print("=== GFNet geometry (sign-swap chimera) ===")
gf_geom = geometry_table(gf, gf_capture, is_complex=False)
print("=== PRISM geometry (phase-swap chimera) ===")
prism_geom = geometry_table(model, prism_capture, is_complex=True)


# ============================================================================
# CELL D2 — MEAN-MAGNITUDE SUBSTITUTION (causal test, per-layer sweep)
# ============================================================================
# Replace |x| with the BATCH-MEAN magnitude pattern; keep own sign/phase.
# Self-label accuracy ~ baseline + low mag_corr  => readout ignores real,
# image-specific magnitude information (H1). If mag_corr was ~1 anyway,
# this substitution changed nothing and H0 stands.

def meanmag_real(x):                        # [B, N, C]
    m = x.abs().mean(dim=0, keepdim=True)
    return torch.sign(x) * m

def meanmag_complex(z):                     # [B, H, W, D] complex
    m = z.abs().mean(dim=0, keepdim=True)
    return m * (z / (z.abs() + EPS))

@torch.no_grad()
def gf_forward_meanmag(model, images, at):
    x = model.patch_embed(images)
    x = model.pos_drop(x + model.pos_embed)
    for i, blk in enumerate(model.blocks):
        if i == at:
            x = meanmag_real(x)
        x = blk(x)
    return model.head(model.norm(x).mean(1))

@torch.no_grad()
def prism_forward_meanmag(model, images, at):
    x, _ = model.rose(images)
    for i, layer in enumerate(model.layers):
        if i == at:
            x = meanmag_complex(x)
        x = layer(x)
    return model.head(model.bridge(model.final_norm(x)).mean(dim=(1, 2)))

def meanmag_sweep(model, fwd, depth, tag):
    rows = []
    for l in range(depth):
        res = run_condition(lambda x, _l=l: fwd(model, x, _l), score_self)
        rows.append(dict(condition=f"{tag}_meanmag_L{l}", **res))
        print(rows[-1])
    return rows

print("=== GFNet mean-magnitude substitution (baseline 78.16) ===")
gf_mm = meanmag_sweep(gf, gf_forward_meanmag, len(gf.blocks), "gfnet")
print("=== PRISM mean-magnitude substitution (baseline 78.07) ===")
prism_mm = meanmag_sweep(model, prism_forward_meanmag, len(model.layers), "prism")


# ============================================================================
# CELL D3 — VERDICT TABLE + FIGURE
# ============================================================================

def verdict(geom, mm, follow_phase_late, name):
    last = geom[-1]
    mm_late = mm[-1]["follow_mag"]          # self-label accuracy, last layer
    print(f"\n--- {name} (last layer) ---")
    print(f"mag_corr(A,B)      = {last['mag_corr']:.3f}   "
          f"(H0 predicts ~1.0)")
    print(f"cos(chimera, B)    = {last['cos_chB']:.3f}   "
          f"(H0 predicts ~1.0; compare cos(A,B) floor = {last['cos_AB']:.3f})")
    print(f"meanmag accuracy   = {mm_late:.1f}%  vs baseline ~78%")
    print(f"follow_phase late  = {follow_phase_late:.1f}%")
    boring = last["mag_corr"] > 0.9 and last["cos_chB"] > 0.95
    if boring:
        print("=> H0-leaning: magnitudes are near-uniform across images; "
              "frame as MAGNITUDE HOMOGENIZATION with depth.")
    elif mm_late > 60 and last["mag_corr"] < 0.75:
        print("=> H1: magnitudes carry image-specific structure the readout "
              "demonstrably ignores; frame as SIGN/PHASE-ROBUST IDENTITY CODE.")
    else:
        print("=> Mixed/depth-dependent: report both metrics per layer; "
              "the truth is the curve, not a verdict.")

verdict(gf_geom, gf_mm, 74.7, "GFNet")
verdict(prism_geom, prism_mm, 75.9, "PRISM2D")

def diagnostic_figure(gf_geom, prism_geom, gf_mm, prism_mm, cfg=OL_CFG):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for geom, name, mk in [(gf_geom, "GFNet", "s"), (prism_geom, "PRISM", "o")]:
        axes[0].plot([r["mag_corr"] for r in geom], mk + "-", label=name)
        axes[1].plot([r["cos_chB"] for r in geom], mk + "-",
                     label=f"{name}: cos(chim,B)")
        axes[1].plot([r["cos_AB"] for r in geom], mk + "--", alpha=0.5,
                     label=f"{name}: cos(A,B) floor")
    axes[0].set_title("Magnitude pattern correlation across images\n"
                      "(H0 predicts -> 1 with depth)")
    axes[1].set_title("Chimera geometry\n(H0 predicts cos(chim,B) -> 1)")

    for mm, name, mk in [(gf_mm, "GFNet", "s"), (prism_mm, "PRISM", "o")]:
        axes[2].plot([r["follow_mag"] for r in mm], mk + "-", label=name)
    axes[2].axhline(78, ls=":", c="gray", label="baseline")
    axes[2].set_title("Mean-magnitude substitution accuracy\n"
                      "(image-specific magnitude info deleted)")

    for ax in axes:
        ax.set_xlabel("layer l"); ax.legend(fontsize=7)
    axes[0].set_ylim(0, 1); axes[1].set_ylim(0, 1); axes[2].set_ylim(0, 100)
    plt.tight_layout()
    out = os.path.join(cfg["run_dir"], "ol_diagnostics.png")
    plt.savefig(out, dpi=200); plt.show()
    print("saved ->", out)

diagnostic_figure(gf_geom, prism_geom, gf_mm, prism_mm)

In [ ]:
# ============================================================================
# GENERALITY: PRETRAINED RESNET-50 & VIT-B/16 — APPEND TO THE SAME SESSION
# ============================================================================
# Requires from earlier cells: OL_CFG, DEVICE, pair_loader, CLASS_NAMES,
# run_condition, score_chimeras, score_self, pair_swap_phase.
# Zero training: torchvision IN-1k weights, logits restricted to the 100
# classes via name mapping (keeps chance at 1%, comparable to PRISM/GFNet).
# Conditions per model: sign swap, stream spectral swap (our FFT basis),
# mean-magnitude substitution. The FFT is the ANALYST'S basis (as in O&L
# 1981) — neither model computes one internally.

import os, csv, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import vit_b_16, ViT_B_16_Weights

EPS = 1e-8

# ============================================================================
# CELL V1 — MODELS + IN-100 -> IN-1k LABEL MAPPING
# ============================================================================

rn_weights, vit_weights = ResNet50_Weights.IMAGENET1K_V2, ViT_B_16_Weights.IMAGENET1K_V1
rn = resnet50(weights=rn_weights).to(DEVICE).eval()
vit = vit_b_16(weights=vit_weights).to(DEVICE).eval()

def build_label_map(class_names, categories):
    """Map IN-100 names -> IN-1k indices. HF names are WordNet synonym
    lists ('Doberman, Doberman pinscher'); torchvision uses one name.
    Try every synonym, normalized, until one hits."""
    norm = lambda s: s.strip().lower().replace("_", " ")
    cat_idx = {norm(c): i for i, c in enumerate(categories)}
    mapping, missing = [], []
    for nm in class_names:
        hit = -1
        for syn in nm.split(","):
            hit = cat_idx.get(norm(syn), -1)
            if hit >= 0:
                break
        mapping.append(hit)
        if hit < 0:
            missing.append(nm)
    if missing:
        print(f"UNMATCHED ({len(missing)}): {missing}")
    return torch.tensor(mapping)

MAP100 = build_label_map(CLASS_NAMES, rn_weights.meta["categories"]).to(DEVICE)
assert (MAP100 >= 0).all(), "resolve unmatched class names first"

def restrict(logits1k):
    return logits1k[:, MAP100]            # [B, 100], argmax -> 0..99


# ============================================================================
# CELL V2 — INTERVENTION PRIMITIVES (real streams, any layout)
# ============================================================================

def sign_swap_any(x):
    """Pairs (2i,2i+1): out[2i] = |A| * sign(B). Works on [B, ...] real."""
    s, m = torch.sign(x), x.abs()
    s_sw = torch.empty_like(s)
    s_sw[0::2], s_sw[1::2] = s[1::2], s[0::2]
    return m * s_sw

def meanmag_any(x):
    m = x.abs().mean(dim=0, keepdim=True)
    return torch.sign(x) * m

def spectral_swap_grid(x, hw_dims):
    """Our Fourier basis: fft2 over spatial dims, phase swap, inverse."""
    f = torch.fft.fft2(x.float(), dim=hw_dims, norm="ortho")
    return torch.fft.ifft2(pair_swap_phase(f), dim=hw_dims, norm="ortho").real

MODES = dict(sign=sign_swap_any, meanmag=meanmag_any)


# ============================================================================
# CELL V3 — RESNET-50 RUNNER (intervene after each residual block; 16 points)
# ============================================================================

def rn_blocks(model):
    stem = nn.Sequential(model.conv1, model.bn1, model.relu, model.maxpool)
    blocks = (list(model.layer1) + list(model.layer2)
              + list(model.layer3) + list(model.layer4))
    return stem, blocks

@torch.no_grad()
def rn_forward_swap(model, images, at, mode):
    stem, blocks = rn_blocks(model)
    x = stem(images)
    for i, blk in enumerate(blocks):
        if i == at:
            if mode == "spectral":
                x = spectral_swap_grid(x, (-2, -1))   # [B, C, H, W]
            else:
                x = MODES[mode](x)
        x = blk(x)
    x = model.avgpool(x).flatten(1)
    return restrict(model.fc(x))

RN_DEPTH = len(rn_blocks(rn)[1])


# ============================================================================
# CELL V4 — VIT-B/16 RUNNER (12 blocks; CLS handling documented)
# ============================================================================
# sign / meanmag: applied to ALL tokens including CLS.
# spectral: patch tokens only (reshaped to 14x14 grid); CLS untouched.
# CAVEAT for interpretation: readout is CLS-only. Late-layer swaps can fail
# via readout locality (CLS already encodes A), not phase failure.

@torch.no_grad()
def vit_forward_swap(model, images, at, mode):
    x = model._process_input(images)                  # [B, N, C] patches
    n = x.shape[0]
    cls = model.class_token.expand(n, -1, -1)
    x = torch.cat([cls, x], dim=1)
    x = x + model.encoder.pos_embedding
    for i, blk in enumerate(model.encoder.layers):
        if i == at:
            if mode == "spectral":
                B, N1, C = x.shape
                g = int(math.sqrt(N1 - 1))
                patches = x[:, 1:].reshape(B, g, g, C)
                patches = spectral_swap_grid(patches, (1, 2))
                x = torch.cat([x[:, :1], patches.reshape(B, g * g, C)], dim=1)
            else:
                x = MODES[mode](x)
        x = blk(x)
    x = model.encoder.ln(x)
    return restrict(model.heads(x[:, 0]))

VIT_DEPTH = len(vit.encoder.layers)


# ============================================================================
# CELL V5 — SWEEPS
# ============================================================================

@torch.no_grad()
def baseline_acc(fwd_plain, name):
    res = run_condition(fwd_plain, score_self)
    print(dict(condition=f"{name}_baseline", **res))
    return dict(condition=f"{name}_baseline", **res)

def sweep(name, fwd, depth, mode, scorer):
    rows = []
    for l in range(depth):
        res = run_condition(lambda x, _l=l: fwd(x, _l), scorer)
        rows.append(dict(condition=f"{name}_{mode}_L{l}", **res))
        print(rows[-1])
    return rows

def generality_experiment(cfg=OL_CFG):
    rows = []
    # ---- ResNet-50 ----
    rows.append(baseline_acc(lambda x: restrict(rn(x)), "rn50"))
    for mode, scorer in [("sign", score_chimeras),
                         ("spectral", score_chimeras),
                         ("meanmag", score_self)]:
        rows += sweep("rn50", lambda x, l, _m=mode:
                      rn_forward_swap(rn, x, l, _m), RN_DEPTH, mode, scorer)
    # ---- ViT-B/16 ----
    rows.append(baseline_acc(lambda x: restrict(vit(x)), "vitb16"))
    for mode, scorer in [("sign", score_chimeras),
                         ("spectral", score_chimeras),
                         ("meanmag", score_self)]:
        rows += sweep("vitb16", lambda x, l, _m=mode:
                      vit_forward_swap(vit, x, l, _m), VIT_DEPTH, mode, scorer)

    out = os.path.join(cfg["run_dir"], "ol_results_generality.csv")
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print("saved ->", out)
    return rows

gen_rows = generality_experiment()


# ============================================================================
# CELL V6 — FOUR-MODEL FIGURE (sign-swap follow_phase across depth, normalized)
# ============================================================================
# Depths differ (10/12/16/12), so plot vs fractional depth l/(L-1).

def four_model_figure(prism_rows, gfnet_rows, gen_rows, cfg=OL_CFG):
    import matplotlib.pyplot as plt
    def fp(rows, prefix):
        sel = [r for r in rows if r["condition"].startswith(prefix)]
        y = [r["follow_phase"] for r in sel]
        x = [i / max(1, len(y) - 1) for i in range(len(y))]
        return x, y
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for rows_, prefix, label, mk in [
            (prism_rows, "stream_L", "PRISM2D (channel phase)", "o"),
            (gfnet_rows, "sign_swap_L", "GFNet (sign)", "s"),
            (gen_rows, "rn50_sign_L", "ResNet-50 (sign)", "^"),
            (gen_rows, "vitb16_sign_L", "ViT-B/16 (sign)", "d")]:
        x, y = fp(rows_, prefix)
        ax.plot(x, y, mk + "-", label=label)
    ax.axhline(1.0, ls=":", c="gray", label="chance (1%)")
    ax.set_xlabel("fractional depth l / (L-1)")
    ax.set_ylabel("% follow phase/sign donor")
    ax.set_ylim(0, 100); ax.legend(fontsize=8)
    ax.set_title("Identity transplant via phase/sign across architectures")
    plt.tight_layout()
    out = os.path.join(cfg["run_dir"], "ol_four_models.png")
    plt.savefig(out, dpi=200); plt.show()
    print("saved ->", out)

# four_model_figure(rows, gf_rows, gen_rows)